# E5 GRPO continuation — Nguyễn Thị Thùy Dương

Attach Full Bonus Version 2 notebook output and original Road Damage Dataset. Enable Internet and GPU T4. Only E5 trains; E0–E4 checkpoints/results are reused. Undefined telemetry is recorded as null; invalid loss/gradient/reward fails explicitly. Checkpoint every five steps.

In [ ]:
import base64, io, os, sys, zipfile, subprocess
from pathlib import Path
ROOT = Path('/kaggle/working/lab_day10_e5')
ROOT.mkdir(exist_ok=True)
os.environ['HF_HOME'] = '/kaggle/temp/day10_hf_cache'
SOURCE_B64 = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_B64))) as archive:
    archive.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-student.txt'], check=True)
from solution.colab_environment import prepare
prepare()
import torch
assert torch.cuda.is_available(), 'Enable GPU T4 and Internet'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'student_tests/test_training_log.py', 'student_tests/test_core.py'], check=True)


In [ ]:
import json
previous_roots = [p.parent.parent for p in Path('/kaggle/input').rglob('selection.json') if p.parent.name == 'report']
assert len(previous_roots) == 1, f'Attach Full Bonus notebook Version 2 output: {previous_roots}'
PREVIOUS = previous_roots[0]
print('PREVIOUS_OUTPUT=', PREVIOUS)
raw_candidates = [p.parent for p in Path('/kaggle/input').rglob('labels-YOLO') if p.is_dir() and (p.parent/'images').is_dir()]
if raw_candidates:
    data_command = ['--source', str(raw_candidates[0])]
else:
    data_command = ['--zip', 'https://www.kaggle.com/api/v1/datasets/download/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes']
subprocess.run([sys.executable, 'scripts/prepare_data.py', *data_command], check=True)
assert json.loads(Path('data/splits/meta.json').read_text())['dataset_version'] == 'd10-21e4070301'
subprocess.run([sys.executable, '-u', '-m', 'solution.finish_e5', '--previous', str(PREVIOUS)], check=True)


In [ ]:
print(Path('REPORT.md').read_text())
print('DOWNLOAD:', ROOT/'day10_e5_results.zip')
print(Path('report/evidence/e5.json').read_text())
